# 06. 점수 기반 평가자 (Score-Based Critic): 수치 점수로 RAG 품질-비용을 제어하기

> 지금까지의 그레이더는 모두 "yes/no"였어요. 이번에는 **0.0~1.0 점수**로 답변 품질을 평가하고, **임계값과 반복 한도**로 품질과 비용의 균형을 잡는 Critic 루프를 만들어 봐요.

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요:

1. **CriticScore** Pydantic 모델로 groundedness/relevance/completeness 세 축의 **float 점수 평가자**를 만들 수 있어요
2. `PASS_THRESHOLD`와 `MAX_ITERATIONS` 상수로 **품질-비용 트레이드오프**를 명시적으로 제어할 수 있어요
3. `add_conditional_edges`의 **path_map**을 활용해 "가장 낮은 점수 축"에 따라 재검색/재생성으로 분기하는 라우터를 구현할 수 있어요
4. 수동 반복 카운터와 `recursion_limit`의 **이중 안전망 구조**를 설명할 수 있어요
5. 이진(binary) 평가와 수치(score) 평가의 **트레이드오프**(leniency bias, 점수 인플레이션 vs 세밀한 제어)를 설명할 수 있어요

## 사전 지식

- `08_RAG/03-CRAG-Self-RAG.ipynb`에서 배운 PDF Retriever, RAG Chain, 이진 그레이더(`GradeDocuments`), `GraphRecursionError` 처리
- `08_RAG/04-Adaptive-RAG.ipynb`에서 배운 조건부 라우팅과 path_map 패턴
- Pydantic `BaseModel` + `with_structured_output()` 기본 사용법

## 왜 점수 기반 평가자인가요?

03장(CRAG/Self-RAG)과 04장(Adaptive RAG)의 그레이더는 모두 **이진(binary) 평가**였어요. "관련 있나? yes/no", "환각인가? yes/no"처럼요. 공식 문서의 agentic-rag 예제도 이진 평가를 사용해요.

하지만 실무의 Agentic RAG 파이프라인에서는 **"얼마나 좋은가"를 수치로 평가하는 Critic 노드** 변형이 흔해요. 답변을 세 축으로 채점하고, 임계값(threshold)을 넘지 못하면 가장 약한 축을 보완하는 방향으로 루프를 도는 방식이에요. 이 노트북은 그 변형을 직접 구현해요.

> 🔑 **핵심 개념**: 이진 평가가 **합격/불합격 시험**이라면, 점수 기반 평가는 **채점표가 있는 논술 시험**이에요. "불합격"만 알면 어디를 고쳐야 할지 모르지만, "근거 0.9 / 관련성 0.9 / 완성도 0.6"이라는 채점표를 받으면 **완성도를 보완**해야 한다는 구체적인 다음 행동이 나와요.

### 아키텍처: Critic 루프

```mermaid
flowchart TD
    START([START]) --> retrieve[retrieve<br/>PDF 벡터 검색]
    retrieve --> generate[generate<br/>답변 생성]
    generate --> critic[critic<br/>3축 점수 평가]
    critic -->|모두 임계값 통과<br/>또는 반복 한도 도달| formatter[formatter<br/>답변 + 점수 카드]
    critic -->|groundedness/relevance 최저| rewrite[rewrite_query<br/>쿼리 재작성]
    critic -->|completeness 최저| generate
    rewrite --> retrieve
    formatter --> END([END])
```

- **검색이 문제**(근거 부족, 관련성 낮음) → 쿼리를 다시 써서 **재검색**
- **생성이 문제**(완성도 낮음) → 같은 문서로 **재생성**
- **통과 또는 한도 도달** → 점수 카드와 함께 종료

### 이진(binary) vs 점수(score): 트레이드오프

| 비교 항목 | 이진 평가 (03·04장) | 점수 평가 (이 노트북) |
|----------|--------------------|--------------------|
| 출력 형식 | `"yes"` / `"no"` | float 0.0~1.0 × 3축 |
| 분기 가능성 | 2갈래 | 임계값·최저 축에 따라 다단계 분기 |
| 보정(calibration) | 쉬움 — 경계가 명확해요 | 어려움 — 같은 답변도 실행마다 0.78↔0.85로 흔들릴 수 있어요 |
| 점수 인플레이션 | 거의 없음 | LLM이 후하게 주는 **leniency bias**가 알려져 있어요 |
| 공식 예시 | 공식 문서 agentic-rag가 이진 사용 | 공식 예시는 없고, 같은 패턴(`with_structured_output`)의 **확장 적용**이에요 |
| 적합한 상황 | 단순 통과/차단 게이트 | "어느 축이 약한가"에 따라 다른 보완 전략이 필요할 때 |

> ⚠️ **자주 하는 실수 1 — 임계값 과신**: LLM 점수는 절대적인 측정값이 아니에요. `PASS_THRESHOLD=0.85`는 "85% 정확하다"가 아니라 **"이 프롬프트·이 모델 조합에서 경험적으로 좋은 답변이 받는 수준"**이에요. 같은 답변을 여러 번 채점하면 점수가 흔들리므로(분산), 임계값은 실제 데이터로 튜닝해야 해요.
>
> ⚠️ **자주 하는 실수 2 — 자기 평가 편향(Self-Evaluation Bias)**: 이 노트북처럼 **답변을 생성한 LLM과 같은 모델이 심판을 보면** 자기가 쓴 답에 후한 점수를 주는 경향이 있어요. 실무에서는 심판용으로 더 강한 모델(예: gpt-4o)이나 다른 계열 모델을 쓰는 경우가 많아요. 교재에서는 비용 때문에 같은 gpt-4o-mini를 쓰지만, 이 한계를 꼭 기억하세요.

### 12_Testing의 LLM-as-Judge와는 뭐가 다른가요?

| 구분 | 이 노트북 (인라인 Critic) | 12_Testing의 LLM-as-Judge |
|------|--------------------------|---------------------------|
| 실행 시점 | **그래프 안**, 답변이 사용자에게 가기 **전** | **오프라인**, 배포 전/후 평가 데이터셋 위에서 |
| 목적 | 런타임 품질 **제어** (낮으면 즉시 보완 루프) | 시스템 품질 **측정** (회귀 감지, A/B 비교) |
| 비용 | 요청마다 발생 → 반복 한도가 필수예요 | 평가 실행 시에만 발생 |
| 도구 | `with_structured_output` + 조건부 엣지 | LangSmith evaluator, openevals |

두 가지는 경쟁 관계가 아니라 **상호 보완**이에요. 인라인 Critic의 임계값을 얼마로 둘지는 오프라인 평가로 결정하는 식이죠.

## 1. 환경 설정

In [ ]:
# ---------------------------------------------------
# 환경 설정
# ---------------------------------------------------
# dotenv: .env 파일에서 API 키를 읽어와요
from dotenv import load_dotenv

# API 키 정보 로드 (OPENAI_API_KEY 등)
load_dotenv(override=True)

In [ ]:
# ---------------------------------------------------
# LangSmith 추적 설정
# ---------------------------------------------------
# LangSmith를 사용하면 Critic 루프가 몇 번 돌았는지 시각적으로 추적할 수 있어요
import os

# os.environ["LANGCHAIN_TRACING_V2"] = "true"
# os.environ["LANGCHAIN_PROJECT"] = "LangGraph-RAG-Score-Critic"

## 2. PDF Retriever와 RAG 체인 재구성 (03장 복습)

`03-CRAG-Self-RAG.ipynb`와 **완전히 동일한 설정**으로 Retriever와 RAG 체인을 다시 만들어요. 이번 노트북의 새로운 내용은 다음 섹션의 Critic이므로, 이 부분은 빠르게 지나가도 좋아요.

**실습 문서**: 소프트웨어정책연구소(SPRi) AI Brief 2023년 12월호  
파일명: `data/SPRI_AI_Brief_2023년12월호_F.pdf`

In [ ]:
# ---------------------------------------------------
# PDF 로더, 임베딩, 벡터스토어 설정 (03장과 동일)
# ---------------------------------------------------
import os
os.environ["KMP_DUPLICATE_LIB_OK"] = "TRUE"

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# PDF 파일 로드 (SPRI AI Brief 2023년 12월호)
PDF_PATH = "data/SPRI_AI_Brief_2023년12월호_F.pdf"

# 페이지 단위로 문서 로드
loader = PyPDFLoader(PDF_PATH)
pages = loader.load()
print(f"총 {len(pages)}페이지 로드 완료")

# 청크로 분할 (03장과 동일한 설정)
splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=50,
)
docs = splitter.split_documents(pages)
print(f"총 {len(docs)}개 청크 생성")

# OpenAI 임베딩 + FAISS 벡터스토어 생성
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vectorstore = FAISS.from_documents(docs, embeddings)

# Retriever 생성 (유사도 기반, 상위 4개 문서 반환)
pdf_retriever = vectorstore.as_retriever(search_kwargs={"k": 4})
# Retriever 준비 완료

In [ ]:
# ---------------------------------------------------
# RAG 체인 구성 (답변 생성용, 03장과 동일)
# ---------------------------------------------------
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# 기본 모델: gpt-4o-mini (비용 효율)
# 다른 모델 옵션: "anthropic:claude-sonnet-4-5", "ollama:llama3"
llm = init_chat_model("openai:gpt-4o-mini", temperature=0)

# RAG 프롬프트 템플릿 정의
RAG_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """You are a helpful assistant. Answer the question based on the provided context.
If the context doesn't contain enough information, say so clearly.
Always cite your sources by mentioning the document name and page number."""),
    ("human", """Context: {context}

Question: {question}

Answer in Korean:""")
])


def format_docs(docs):
    """문서 리스트를 프롬프트 입력 형식으로 포맷팅해요."""
    return "\n\n".join(
        f"<document>\n<content>{doc.page_content}</content>\n"
        f"<source>{doc.metadata.get('source', 'unknown')}</source>\n"
        f"<page>{doc.metadata.get('page', 0) + 1}</page>\n</document>"
        for doc in docs
    )


# RAG 체인: 프롬프트 → LLM → 텍스트 파서
rag_chain = RAG_PROMPT | llm | StrOutputParser()
# RAG 체인 준비 완료

## 3. 점수 기반 평가자(Critic) 구성

### 3-1. CriticScore 모델

03장의 `GradeDocuments`는 `binary_score: str` 필드 하나였어요. 이번에는 **float 필드 3개 + 근거 문자열**로 확장해요. 패턴은 똑같이 `BaseModel` + `with_structured_output()`이에요.

세 가지 평가 축:

| 축 | 질문 | 낮을 때의 의미 |
|----|------|---------------|
| **groundedness** (근거성) | 답변이 검색된 문서에 근거하는가? | 환각 가능성 → 더 좋은 문서가 필요해요 (**재검색**) |
| **relevance** (관련성) | 답변이 질문의 핵심을 다루는가? | 검색 방향이 어긋남 → 쿼리 재작성 후 **재검색** |
| **completeness** (완성도) | 질문의 모든 부분에 충분히 답했는가? | 문서는 충분한데 생성이 부실 → **재생성** |

> 💡 **실무 팁**: `Field(ge=0.0, le=1.0)`은 Pydantic이 범위를 **검증**하게 해주고, description은 LLM에게 채점 기준을 **알려주는 프롬프트** 역할도 해요. description을 구체적으로 쓸수록 점수가 안정돼요.

In [ ]:
# ---------------------------------------------------
# CriticScore: 3축 float 점수 평가 모델
# ---------------------------------------------------
from pydantic import BaseModel, Field


class CriticScore(BaseModel):
    """생성된 답변을 세 축의 수치 점수로 평가해요."""

    groundedness: float = Field(
        ge=0.0, le=1.0,
        description="답변이 제공된 문서에 근거하는 정도. "
                    "1.0 = 모든 문장이 문서로 뒷받침됨, 0.0 = 문서와 무관한 내용",
    )
    relevance: float = Field(
        ge=0.0, le=1.0,
        description="답변이 질문의 핵심 의도를 다루는 정도. "
                    "1.0 = 질문에 정확히 답함, 0.0 = 동문서답",
    )
    completeness: float = Field(
        ge=0.0, le=1.0,
        description="질문의 모든 부분에 충분히 답한 정도. "
                    "1.0 = 빠진 내용 없음, 0.0 = 핵심 내용 누락",
    )
    rationale: str = Field(
        description="세 점수를 그렇게 매긴 근거를 한국어 2~3문장으로 설명",
    )


# 구조화된 출력을 사용하는 Critic LLM
# with_structured_output: 항상 CriticScore 형식(Pydantic 객체)으로 응답해요
critic_llm = llm.with_structured_output(CriticScore)

# Critic 프롬프트: 채점 기준(rubric)을 명시할수록 점수가 안정돼요
critic_system = """You are a strict critic evaluating a RAG answer.
Score three dimensions from 0.0 to 1.0:
- groundedness: Is every claim in the answer supported by the provided documents?
- relevance: Does the answer address the core intent of the question?
- completeness: Does the answer cover all parts of the question sufficiently?

Be strict. Reserve scores above 0.9 for flawless answers.
If the documents do not contain the needed information and the answer admits it,
groundedness can be high but completeness must be low."""

critic_prompt = ChatPromptTemplate.from_messages([
    ("system", critic_system),
    ("human", """Question:\n{question}\n\nRetrieved documents:\n{context}\n\nGenerated answer:\n{generation}"""),
])

# Critic 체인: 프롬프트 → 구조화 출력 LLM
critic_chain = critic_prompt | critic_llm
# Critic 체인 준비 완료

### 3-2. Critic 단독 테스트

그래프에 넣기 전에 Critic이 어떤 식으로 채점하는지 단독으로 확인해 봐요. 결과는 dict가 아니라 **Pydantic 객체**이므로 `result.groundedness`처럼 **필드로 접근**해야 해요.

In [ ]:
# ---------------------------------------------------
# Critic 단독 테스트
# ---------------------------------------------------
# 일부러 '문서에 근거하지만 불완전한 답변'을 채점시켜 봐요
test_docs = pdf_retriever.invoke("삼성전자가 개발한 생성형 AI")
test_answer = "삼성전자는 생성형 AI를 개발했습니다."  # 이름(삼성 가우스)이 빠진 불완전한 답변

result = critic_chain.invoke({
    "question": "삼성전자가 개발한 생성형 AI의 이름은?",
    "context": format_docs(test_docs),
    "generation": test_answer,
})

# 결과는 CriticScore Pydantic 객체예요 (dict 아님!)
print(f"groundedness : {result.groundedness:.2f}")
print(f"relevance    : {result.relevance:.2f}")
print(f"completeness : {result.completeness:.2f}")
print(f"근거: {result.rationale}")

## 4. 그래프 상태와 제어 상수 정의

상태에는 03장의 `question`/`generation`/`documents`에 더해 두 가지가 추가돼요:

- **`iteration`**: 수동 반복 카운터. Critic이 채점할 때마다 1씩 증가하고, `MAX_ITERATIONS`에 도달하면 점수가 낮아도 종료해요
- **`score_history`**: 매 반복의 점수를 기록해요. formatter가 마지막에 점수 카드로 보여줘요

### recursion_limit과 수동 카운터의 관계

03장에서는 `recursion_limit` 하나로 무한 루프를 막았어요. 이번에는 **이중 안전망** 구조를 써요:

| 장치 | 역할 | 동작 |
|------|------|------|
| `iteration` + `MAX_ITERATIONS` | **내부 제어** (정상 종료) | 한도 도달 시 formatter로 라우팅 → 점수 카드와 함께 **답변을 반환**해요 |
| `recursion_limit` (config) | **외부 안전망** (비정상 종료) | 라우팅 함수에 버그가 있어도 `GraphRecursionError`로 강제 중단해요 |

`recursion_limit`은 `config={"recursion_limit": N}`처럼 **config 최상위**에 넣어야 해요. `configurable` 안에 넣으면 조용히 무시되니 주의하세요. 안전망 값은 `MAX_ITERATIONS × 루프 노드 수 + 여유`로 잡으면 돼요.

> 💡 **참고**: LangGraph는 수동 카운터 대신 `from langgraph.managed import RemainingSteps`라는 **자동 감소 매니지드 값**도 제공해요. 실무 코드에서 자주 보이지만, 이 노트북에서는 동작이 눈에 보이는 수동 카운터로 학습해요.

In [ ]:
# ---------------------------------------------------
# 그래프 상태(State)와 제어 상수 정의
# ---------------------------------------------------
from typing import Annotated, List
from typing_extensions import TypedDict


class CriticRAGState(TypedDict):
    """Score-Based Critic RAG 그래프의 상태 정의예요."""

    question: Annotated[str, "현재 질문 (재작성될 수 있어요)"]
    original_question: Annotated[str, "사용자의 원본 질문"]
    generation: Annotated[str, "생성된 답변"]
    documents: Annotated[List, "검색된 문서 리스트"]
    iteration: Annotated[int, "Critic 평가 횟수 (수동 카운터)"]
    score_history: Annotated[List[dict], "매 반복의 점수 기록"]
    final_report: Annotated[str, "최종 답변 + 점수 카드"]


# ---------------------------------------------------
# 품질-비용 제어 상수
# ---------------------------------------------------
# PASS_THRESHOLD: 세 축 모두 이 값 이상이어야 통과해요
#   - 높이면: 품질 ↑, 반복(비용·지연) ↑
#   - 낮추면: 비용 ↓, 부실한 답변 통과 위험 ↑
PASS_THRESHOLD = 0.85

# MAX_ITERATIONS: Critic 평가 최대 횟수 (내부 제어)
MAX_ITERATIONS = 3

# RECURSION_LIMIT: 외부 안전망 (버그 대비)
# 한 루프에 최대 4개 노드(rewrite→retrieve→generate→critic)가 돌아요
RECURSION_LIMIT = MAX_ITERATIONS * 4 + 6
print(f"PASS_THRESHOLD={PASS_THRESHOLD}, MAX_ITERATIONS={MAX_ITERATIONS}, RECURSION_LIMIT={RECURSION_LIMIT}")

## 5. 노드 함수 정의

### 5-1. retrieve · generate 노드

03장과 거의 같지만, `retrieve`가 첫 호출일 때 `original_question`과 카운터 초기값을 함께 채워요.

In [ ]:
# ---------------------------------------------------
# 노드 1: retrieve - 문서 검색
# ---------------------------------------------------


def retrieve(state: CriticRAGState) -> CriticRAGState:
    """문서를 검색하는 노드예요. 첫 실행 시 상태를 초기화해요."""
    # ==== [RETRIEVE] ====
    question = state["question"]

    # 벡터 검색으로 관련 문서 찾기
    documents = pdf_retriever.invoke(question)

    return {
        "documents": documents,
        # 첫 실행이면 원본 질문과 카운터를 초기화해요 (이미 있으면 유지)
        "original_question": state.get("original_question") or question,
        "iteration": state.get("iteration", 0),
        "score_history": state.get("score_history", []),
    }


# ---------------------------------------------------
# 노드 2: generate - 답변 생성
# ---------------------------------------------------


def generate(state: CriticRAGState) -> CriticRAGState:
    """검색된 문서를 기반으로 답변을 생성하는 노드예요.

    재생성(retry_generate)으로 다시 들어온 경우에는 직전 Critic의
    rationale(채점 근거)을 프롬프트에 함께 넣어요. temperature=0 모델은
    같은 입력에 (거의) 같은 출력을 내기 때문에, 피드백을 입력에 반영하지
    않으면 재생성 루프가 똑같은 답변만 반복하는 무의미한 루프가 돼요.
    """
    # ==== [GENERATE] ====
    question = state["question"]
    documents = state["documents"]

    context = format_docs(documents) if documents else "관련 문서를 찾을 수 없습니다."

    # 이전 채점 기록이 있으면 = 재생성 경로 → Critic 피드백을 질문에 반영해요
    score_history = state.get("score_history") or []
    if score_history:
        rationale = score_history[-1]["rationale"]
        question = (
            f"{question}\n\n"
            f"[이전 답변에 대한 평가 피드백]\n{rationale}\n"
            f"위 피드백에서 지적된 부족한 부분을 보완하여, "
            f"이전보다 더 완전하고 문서에 근거한 답변을 작성하세요."
        )

    generation = rag_chain.invoke({"context": context, "question": question})
    return {"generation": generation}

### 5-2. critic 노드

이 그래프의 심장이에요. 답변을 3축으로 채점하고, **카운터를 증가**시키고, 점수를 **기록**해요. 라우팅 판단은 노드가 아니라 다음의 조건부 엣지가 담당해요 — 노드는 상태를 바꾸고, 엣지는 경로를 고른다는 역할 분리를 기억하세요.

In [ ]:
# ---------------------------------------------------
# 노드 3: critic - 3축 점수 평가
# ---------------------------------------------------


def critic(state: CriticRAGState) -> CriticRAGState:
    """생성된 답변을 CriticScore로 채점하는 노드예요."""
    # ==== [CRITIC] ====
    question = state["original_question"]  # 채점 기준은 항상 원본 질문이에요
    documents = state["documents"]
    generation = state["generation"]

    context = format_docs(documents) if documents else "(검색된 문서 없음)"

    # 3축 채점 (결과는 CriticScore Pydantic 객체)
    score = critic_chain.invoke({
        "question": question,
        "context": context,
        "generation": generation,
    })

    iteration = state.get("iteration", 0) + 1
    record = {
        "iteration": iteration,
        "groundedness": score.groundedness,
        "relevance": score.relevance,
        "completeness": score.completeness,
        "rationale": score.rationale,
    }

    print(f"  [{iteration}회차 채점] G={score.groundedness:.2f} "
          f"R={score.relevance:.2f} C={score.completeness:.2f}")

    return {
        "iteration": iteration,
        "score_history": state.get("score_history", []) + [record],
    }

### 5-3. rewrite_query 노드

근거성/관련성이 낮을 때 = **검색이 문제**일 때 실행돼요. 03장의 Query Rewriter와 같은 패턴인데, 직전 점수의 `rationale`을 힌트로 넘겨서 "왜 부족했는지"를 반영한 재작성을 해요.

In [ ]:
# ---------------------------------------------------
# 노드 4: rewrite_query - 쿼리 재작성 (재검색 준비)
# ---------------------------------------------------
rewrite_system = """You are a question re-writer that converts an input question to a better version
that is optimized for vectorstore retrieval.
You are given critic feedback explaining why the previous answer scored low.
Use it to reformulate the question. Output only the improved question, no explanation."""

rewrite_prompt = ChatPromptTemplate.from_messages([
    ("system", rewrite_system),
    ("human", "Original question:\n{question}\n\nCritic feedback:\n{feedback}\n\nFormulate an improved question."),
])

# 질문 재작성기: 프롬프트 → LLM → 텍스트 파서
question_rewriter = rewrite_prompt | llm | StrOutputParser()


def rewrite_query(state: CriticRAGState) -> CriticRAGState:
    """Critic 피드백을 반영하여 질문을 재작성하는 노드예요."""
    # ==== [REWRITE QUERY] ====
    question = state["original_question"]
    last_score = state["score_history"][-1]

    better_question = question_rewriter.invoke({
        "question": question,
        "feedback": last_score["rationale"],
    })
    print(f"  재작성: {better_question}")
    return {"question": better_question}

### 5-4. formatter 노드

루프가 끝나면(통과 또는 한도 도달) 최종 답변과 함께 **점수 카드**를 만들어요. 실무에서 점수 카드는 로그·모니터링 대시보드로 보내는 경우가 많아요 — 사용자에게는 답변만, 운영자에게는 점수를 보여주는 식이죠.

In [ ]:
# ---------------------------------------------------
# 노드 5: formatter - 최종 답변 + 점수 카드
# ---------------------------------------------------


def formatter(state: CriticRAGState) -> CriticRAGState:
    """최종 답변과 점수 카드를 합쳐 리포트를 만드는 노드예요."""
    # ==== [FORMATTER] ====
    last = state["score_history"][-1]
    passed = all(
        last[axis] >= PASS_THRESHOLD
        for axis in ("groundedness", "relevance", "completeness")
    )
    status = "✅ 통과" if passed else f"⚠️ 반복 한도 도달 (점수 미달인 채 종료)"

    # 점수 카드 구성
    lines = [
        "=" * 50,
        f"📋 점수 카드  |  상태: {status}",
        f"질문: {state['original_question']}",
        "-" * 50,
    ]
    for rec in state["score_history"]:
        lines.append(
            f"  {rec['iteration']}회차: "
            f"근거성 {rec['groundedness']:.2f} / "
            f"관련성 {rec['relevance']:.2f} / "
            f"완성도 {rec['completeness']:.2f}"
        )
    lines += [
        "=" * 50,
        "",
        "💬 최종 답변:",
        state["generation"],
    ]
    report = "\n".join(lines)
    return {"final_report": report}

### 5-5. 조건부 엣지: route_after_critic

라우팅 규칙을 정리하면:

1. **세 축 모두 `PASS_THRESHOLD` 이상** → `formatter` (통과)
2. **`iteration >= MAX_ITERATIONS`** → `formatter` (한도 도달, 현재 답변으로 종료)
3. **최저 점수 축이 groundedness 또는 relevance** → 검색이 문제 → `rewrite_query` (재검색 경로)
4. **최저 점수 축이 completeness** → 생성이 문제 → `generate` (재생성)

반환값은 `"pass"` / `"retry_retrieve"` / `"retry_generate"` 같은 **의미 있는 키**로 하고, `add_conditional_edges`의 **path_map**으로 실제 노드명에 매핑해요. path_map 없이 노드명을 직접 반환하면 오타가 **컴파일 시점에 안 잡히고 런타임 오류**로 터지니, 분기가 많을수록 path_map이 안전해요.

In [ ]:
# ---------------------------------------------------
# 조건부 엣지: route_after_critic
# ---------------------------------------------------


def route_after_critic(state: CriticRAGState) -> str:
    """Critic 점수에 따라 다음 경로를 결정하는 라우팅 함수예요.

    Returns:
        'pass'           : 통과 또는 반복 한도 도달 → formatter
        'retry_retrieve' : 근거성/관련성 부족 → 쿼리 재작성 후 재검색
        'retry_generate' : 완성도 부족 → 같은 문서 + Critic 피드백으로 재생성
    """
    # ==== [ROUTE AFTER CRITIC] ====
    last = state["score_history"][-1]
    scores = {
        "groundedness": last["groundedness"],
        "relevance": last["relevance"],
        "completeness": last["completeness"],
    }

    # 1) 모든 축이 임계값을 넘으면 통과
    if all(v >= PASS_THRESHOLD for v in scores.values()):
        #   결정: 통과 (모든 축 임계값 이상)
        return "pass"

    # 2) 반복 한도 도달 → 현재 답변으로 종료 (비용 통제)
    if state["iteration"] >= MAX_ITERATIONS:
        #   결정: 한도 도달 → 종료
        return "pass"

    # 3) 가장 낮은 축을 찾아 보완 전략을 선택해요
    weakest = min(scores, key=scores.get)
    if weakest in ("groundedness", "relevance"):
        #   결정: 검색이 문제 → 쿼리 재작성 후 재검색
        return "retry_retrieve"
    else:
        #   결정: 생성이 문제 → 재생성 (generate가 직전 rationale을 프롬프트에 반영해요)
        return "retry_generate"

## 6. 그래프 조립 및 시각화

> 🎯 **강의 포인트**: 시각화를 보면서 세 경로를 설명해요:
> - **통과 경로**: retrieve → generate → critic → formatter
> - **재검색 루프**: critic → rewrite_query → retrieve → generate → critic
> - **재생성 루프**: critic → generate → critic

In [ ]:
# ---------------------------------------------------
# Score-Based Critic 그래프 조립
# ---------------------------------------------------
from langgraph.graph import END, StateGraph, START
from langgraph.checkpoint.memory import MemorySaver

# CriticRAGState 기반 그래프 초기화
workflow = StateGraph(CriticRAGState)

# 노드 등록
workflow.add_node("retrieve", retrieve)
workflow.add_node("generate", generate)
workflow.add_node("critic", critic)
workflow.add_node("rewrite_query", rewrite_query)
workflow.add_node("formatter", formatter)

# 엣지 연결
workflow.add_edge(START, "retrieve")        # 시작 → 검색
workflow.add_edge("retrieve", "generate")   # 검색 → 생성
workflow.add_edge("generate", "critic")     # 생성 → 채점

# 조건부 엣지: 점수에 따라 3갈래 분기 (path_map으로 의미 키 → 노드명 매핑)
workflow.add_conditional_edges(
    "critic",
    route_after_critic,
    {
        "pass": "formatter",             # 통과 또는 한도 도달 → 점수 카드
        "retry_retrieve": "rewrite_query",  # 근거성/관련성 부족 → 재검색 루프
        "retry_generate": "generate",       # 완성도 부족 → 피드백 반영 재생성 루프
    },
)

# 재작성 후 다시 검색 (루프 구성)
workflow.add_edge("rewrite_query", "retrieve")
workflow.add_edge("formatter", END)

# MemorySaver로 체크포인트 저장 (03장 Self-RAG와 동일)
app = workflow.compile(checkpointer=MemorySaver())
# 그래프 컴파일 완료

In [ ]:
# ---------------------------------------------------
# 그래프 시각화
# ---------------------------------------------------
# 그래프 흐름: START → retrieve → generate → critic → (formatter | rewrite_query → retrieve | generate) → END
# retrieve: PDF 벡터 검색으로 관련 문서를 수집해요
# generate: 문서 기반으로 답변을 생성해요
# critic: 3축(근거성/관련성/완성도) float 점수로 채점하고 카운터를 증가시켜요
# route_after_critic: 통과/한도 도달 → formatter, 검색 문제 → rewrite_query, 생성 문제 → generate
# formatter: 최종 답변 + 점수 카드 리포트를 만들어요
from IPython.display import Image, display

display(Image(app.get_graph().draw_mermaid_png()))

## 7. 실행 테스트

두 가지 경우를 테스트해요:

1. **PDF 내 정보** → 첫 채점에서 높은 점수 → 1회차에 통과
2. **PDF에 없는 정보** → 점수 미달 → 루프 반복 → `MAX_ITERATIONS` 도달 후 안전 종료

In [ ]:
# ---------------------------------------------------
# 테스트 1: PDF에 있는 정보 질문 (통과 경로)
# ---------------------------------------------------
import uuid
from langchain_core.runnables import RunnableConfig

# recursion_limit은 config 최상위에! (configurable 안에 넣으면 무시돼요)
config = RunnableConfig(
    recursion_limit=RECURSION_LIMIT,
    configurable={"thread_id": str(uuid.uuid4())},
)

inputs = {"question": "삼성전자가 개발한 생성형 AI의 이름은?"}

# ============================================================
print(f"질문: {inputs['question']}")
# ============================================================

# 스트리밍 실행으로 각 노드의 처리 과정 확인
final_state = None
for chunk in app.stream(inputs, config, stream_mode="updates"):
    for node_name, node_output in chunk.items():
        print(f"\n--- 노드: {node_name} ---")
        if node_output and "final_report" in node_output:
            final_state = node_output

if final_state:
    print("\n" + final_state["final_report"])

**예상 동작 (테스트 1)**: PDF에 "삼성 가우스" 정보가 있으므로 첫 생성부터 좋은 답변이 나와요. Critic이 1회차에 `G=0.95 R=0.95 C=0.90`처럼 세 축 모두 0.85 이상을 주면 `route_after_critic`이 `"pass"`를 반환하고, 곧바로 `formatter`로 가서 1회차 점수 한 줄만 적힌 점수 카드와 최종 답변이 출력돼요.

> 💡 **재현성 주의**: LLM 점수에는 분산이 있어요. 같은 질문이라도 가끔 0.85 직전 점수(예: completeness 0.80)가 나와 재생성 루프를 한 번 돌 수 있어요. 그것도 정상 동작이에요 — 점수 기반 제어의 특성을 눈으로 확인하는 기회로 삼으세요.

In [ ]:
# ---------------------------------------------------
# 테스트 2: PDF에 없는 정보 질문 (루프 → 한도 도달)
# ---------------------------------------------------
# 이 질문의 답은 PDF에 없으므로 점수가 임계값을 못 넘고 루프를 돌아요
config2 = RunnableConfig(
    recursion_limit=RECURSION_LIMIT,
    configurable={"thread_id": str(uuid.uuid4())},
)

inputs2 = {"question": "2025년 한국의 AI 기본법 시행령 주요 내용은 무엇인가요?"}

# ============================================================
print(f"질문: {inputs2['question']}")
# ============================================================

final_state2 = None
for chunk in app.stream(inputs2, config2, stream_mode="updates"):
    for node_name, node_output in chunk.items():
        print(f"\n--- 노드: {node_name} ---")
        if node_output and "final_report" in node_output:
            final_state2 = node_output

if final_state2:
    print("\n" + final_state2["final_report"])

**예상 동작 (테스트 2)**: 2023년 12월호 PDF에는 2025년 시행령 내용이 없어요. 답변은 "문서에 해당 정보가 없습니다" 류가 되고, Critic은 completeness(또는 relevance)에 낮은 점수를 줘요. 그러면:

1. **1회차**: 점수 미달 → 최저 축에 따라 `rewrite_query`(재검색) 또는 `generate`(재생성)로 분기
2. **2회차**: 쿼리를 바꿔도 문서에 없는 건 없으므로 여전히 미달
3. **3회차**: `iteration >= MAX_ITERATIONS` → `"pass"` 강제 반환 → `formatter`

점수 카드에는 3회차의 점수 이력이 모두 찍히고, 상태는 "⚠️ 반복 한도 도달"로 표시돼요. 03장의 `GraphRecursionError`처럼 **예외로 터지는 게 아니라**, 한도 도달을 그래프 안에서 인지하고 **사용자에게 정중한 답변 + 점수 근거를 함께 반환**한다는 점이 이 패턴의 장점이에요. `recursion_limit`은 그 뒤에 서 있는 최후의 안전망일 뿐이고요.

## 실습

### 과제: 가중 평균(Weighted Score) 라우팅으로 바꿔보기

현재 라우터는 "세 축 **모두** 임계값 이상"이라는 엄격한 기준이에요. 실무에서는 축마다 중요도가 다른 경우가 많아요. 예를 들어 **환각이 치명적인 도메인**(의료, 법률)이라면 groundedness에 더 큰 가중치를 줘야겠죠.

다음 요구사항으로 `route_after_critic_weighted`를 구현하고, 그래프를 다시 조립해서 두 테스트 질문의 라우팅 결과가 어떻게 달라지는지 비교해 보세요:

1. 가중치: `WEIGHTS = {"groundedness": 0.5, "relevance": 0.3, "completeness": 0.2}`
2. **가중 평균 점수** `weighted = Σ(점수 × 가중치)`가 `0.85` 이상이면 통과
3. 단, **groundedness가 0.7 미만이면 가중 평균과 무관하게 무조건 재검색** (환각 방어 가드레일 — 평균에 묻히면 안 되는 must-pass 기준이에요)
4. 반복 한도 로직은 기존과 동일하게 유지

힌트: 12_Testing에서 배울 "must-pass 기준은 평균으로 상쇄될 수 없다"는 원칙을 라우터 코드로 옮기는 연습이에요.

In [ ]:
# ---------------------------------------------------
# 실습: 가중 평균 라우터 구현
# ---------------------------------------------------
# TODO: 아래 골격을 완성해 보세요

WEIGHTS = {"groundedness": 0.5, "relevance": 0.3, "completeness": 0.2}
GROUNDEDNESS_FLOOR = 0.7  # must-pass 가드레일


def route_after_critic_weighted(state: CriticRAGState) -> str:
    """가중 평균 + must-pass 가드레일 라우팅 함수예요."""
    last = state["score_history"][-1]

    # TODO 1: 반복 한도 체크 (기존과 동일)

    # TODO 2: groundedness가 GROUNDEDNESS_FLOOR 미만이면
    #         가중 평균과 무관하게 'retry_retrieve' 반환

    # TODO 3: 가중 평균 계산
    # weighted = sum(last[axis] * w for axis, w in WEIGHTS.items())

    # TODO 4: weighted >= 0.85 → 'pass',
    #         아니면 최저 축 기준으로 'retry_retrieve' / 'retry_generate'
    pass


# TODO 5: 새 StateGraph를 만들어 critic의 조건부 엣지만
#         route_after_critic_weighted로 교체하고,
#         두 테스트 질문으로 기존 라우터와 결과를 비교해 보세요

## 핵심 요약

이 노트북에서 다음 내용을 배웠어요:

- **CriticScore**: `BaseModel` + `Field(ge=0.0, le=1.0)` + `with_structured_output()`으로 3축 float 점수 평가자를 만들었어요. 03장 이진 그레이더와 패턴은 같고 출력 형식만 확장한 거예요 (공식 예시는 이진, 이 노트북은 확장 적용).
- **PASS_THRESHOLD + MAX_ITERATIONS**: 임계값으로 품질 기준을, 반복 한도로 비용 상한을 명시해서 **품질-비용 트레이드오프를 코드로 표현**했어요.
- **path_map 라우팅**: `"pass"` / `"retry_retrieve"` / `"retry_generate"` 같은 의미 키를 path_map으로 노드에 매핑하면 분기 의도가 명확해져요. 최저 점수 축에 따라 **재검색**(검색 문제)과 **재생성**(생성 문제)을 구분했어요.
- **이중 안전망**: 수동 `iteration` 카운터가 정상 종료(점수 카드와 함께 반환)를, `recursion_limit`이 비정상 종료(버그 방어)를 담당해요. `recursion_limit`은 config **최상위**에 넣어야 해요.
- **점수 평가의 함정**: leniency bias(후한 채점), 점수 분산, 같은 LLM 심판의 자기 평가 편향을 항상 의심하세요. 임계값은 오프라인 평가(12_Testing의 LLM-as-Judge)로 검증한 뒤 정하는 게 좋아요.

### 5가지 사고 도구로 보면?

이 패턴은 Part 03에서 배운 사고 도구 중 **Verify + Correct**의 조합이에요:

| 사고 도구 | 이 노트북에서의 구현 |
|----------|--------------------|
| **Verify (검증)** | critic 노드가 답변을 3축 수치로 검증해요 — 03장의 yes/no 검증을 "얼마나"로 정밀화한 거예요 |
| **Correct (교정)** | route_after_critic이 **약한 축에 맞는 교정 전략**(재검색 vs 재생성)을 선택해요 |
| Constrain (제약) | (보조) MAX_ITERATIONS·recursion_limit이 교정 루프의 폭주를 제약해요 |

Verify가 정밀해질수록 Correct도 정밀해진다는 것 — 이게 이진 평가에서 점수 평가로 확장한 진짜 이유예요.

## 다음 노트북 예고

다음 `07-Memory-Augmented-RAG.ipynb`에서는 **장기 기억을 RAG 그래프에 통합**해요. 지금까지의 RAG는 매 질문이 백지에서 시작했지만, `07_Memory/02-Long-Term-Memory.ipynb`에서 배운 **Store API**(`InMemoryStore`, 네임스페이스, `put`/`search`)를 RAG 그래프 안으로 가져오면 "이 사용자가 전에 뭘 물었고 뭘 선호하는지"를 기억하는 검색 시스템이 돼요. 7장의 Store API가 다시 등장하고, 8장 RAG 전체 패턴을 묶는 캡스톤 성격의 노트북이니 기대해 주세요!